In [0]:
# ===========================================
#      INSPEKTOR BUDŻET — Notebook 01: Ingest Bronze
# Cel: Utworzenie struktury i wczytanie surowych CSV
# ===========================================

# --- PARAMETRY KONFIGURACYJNE (widgety) ---
# Widgety pozwalają:
#  • Zmienić wartości przez UI na górze notebooka (bez edycji kodu)
#  • Przekazać inne wartości przy uruchamianiu z Jobs
#  • Użyć tego samego notebooka dla różnych projektów/dostawców

dbutils.widgets.text("projekt", "inspektor_budzet")
dbutils.widgets.text("dostawca", "rowkop")

projekt = dbutils.widgets.get("projekt")
dostawca = dbutils.widgets.get("dostawca")

catalog = projekt
schema = dostawca
sciezka_dostawcy = f"/Volumes/{catalog}/{schema}"

print(f"Notebook 01 — Bronze 🥉")
print(f"Projekt:  {projekt} ✅")
print(f"Dostawca: {dostawca} ✅")
print(f"Ścieżka: {sciezka_dostawcy}/...")

# --- TWORZENIE FOLDERÓW DLA NOWEGO DOSTAWCY ---

# Tworzymy katalog i schemat w Unity Catalog
spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")
spark.sql(f"USE CATALOG {catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {schema}")
spark.sql(f"USE SCHEMA {schema}")

# Tworzymy Volume (miejsce na pliki)
spark.sql("CREATE VOLUME IF NOT EXISTS bronze")
spark.sql("CREATE VOLUME IF NOT EXISTS silver")
spark.sql("CREATE VOLUME IF NOT EXISTS gold")
spark.sql("CREATE VOLUME IF NOT EXISTS blueprint")

print("Struktura utworzona ✅")
print(f"  {catalog}.{schema}.bronze   ← surowe CSV")
print(f"  {catalog}.{schema}.silver   ← oczyszczone")
print(f"  {catalog}.{schema}.gold     ← raporty")
print(f"  {catalog}.{schema}.blueprint ← kontrakt JSON")

In [ ]:
# Kopiowanie CSV z ADLS do Volumes (bronze)
storage_account = "inspektorbudzet3"
container = "erp-data"
storage_key = dbutils.secrets.get(scope="AzureADLS", key="storage-key")

from azure.storage.blob import BlobServiceClient

blob_client = BlobServiceClient(
    account_url=f"https://{storage_account}.blob.core.windows.net",
    credential=storage_key
)
container_client = blob_client.get_container_client(container)

vol_path = f"/Volumes/{catalog}/{schema}/bronze"
for blob in container_client.list_blobs():
    if blob.name.endswith(".csv"):
        data = container_client.download_blob(blob.name).readall()
        dbutils.fs.put(f"{vol_path}/{blob.name}", data.decode("utf-8"), overwrite=True)
        print(f"Skopiowano: {blob.name}")

In [0]:
# Sprawdzam co zawiera folder bronze, czy sa tam jakies pliki
files = dbutils.fs.ls(f"{sciezka_dostawcy}/bronze")
print(f"Pliki w bronze: {len(files)}")
for f in files:
    print(f"  {f.name}")

In [0]:
# Wczytaj pliki CSV z folderu bronze
df_bronze = (spark.read
    .option("header", "true")
    .option("sep", ";")
    .option("encoding", "UTF-8")
    .option("inferSchema", "true")
    .csv(f"{sciezka_dostawcy}/bronze/*.csv")
)

print(f"Wczytano {df_bronze.count()} wierszy")
print(f"Kolumny: {df_bronze.columns}")

In [0]:
display(df_bronze)

In [0]:
# Zapisujemy DataFrame jako trwałą tabelę Delta
# Nazwa: {catalog}.{schema}.bronze_usage_internal
nazwa_tabeli = f"{catalog}.{schema}.bronze_usage_internal"

(df_bronze.write
    .mode("overwrite")
    .saveAsTable(nazwa_tabeli)
)

print(f"Tabela zapisana: {nazwa_tabeli} ✅")